# SipSara synthetic datasets and pipeline benchmark (C1-C3)

This notebook generates inspectable, fully synthetic datasets and evaluates candidate models. It does **not** train or replace the deployed runtime artifacts, write to MongoDB, or validate a diagnosis. Synthetic labels are scenario labels defined by provisional simulator rules.

C1 uses generated touch/activity rounds passed through the live C1 learner-feature builder. C2 uses Grade 1 Skill 6 textbook prompts and controlled text edits passed through the live Sinhala alignment service; this measures alignment, **not Whisper/STT speech accuracy**. C3 compares classifiers on paired, noisy synthetic multimodal scenarios using the current 13-feature contract; the values are not actual recordings or outputs from Whisper.

Data-quality/reporting practices are informed by [ISO/IEC 5259-1](https://www.iso.org/standard/81088.html) and [5259-2](https://www.iso.org/standard/81860.html), [NIST SDNist synthetic-data reporting](https://www.nist.gov/services-resources/software/sdnist-synthetic-data-report-tool), and [TRIPOD+AI](https://www.tripod-statement.org/scope/). This is guidance, not certification. Scores below show recovery of generated scenario labels only—not accuracy on real children or clinical validity.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repository root even when Jupyter starts in a nested folder.
here = Path.cwd().resolve()
repo = next((p for p in [here, *here.parents] if (p / 'app/backend/scripts/synthetic_research_benchmark.py').exists()), None)
if repo is None:
    raise FileNotFoundError('Open this notebook from inside the R26-SE-031 repository.')
script = repo / 'app/backend/scripts/synthetic_research_benchmark.py'
out = repo / 'app/backend/research/synthetic_benchmark_v1'
run = subprocess.run([sys.executable, str(script), '--learners-per-class', '80', '--seed', '20261021', '--output-dir', str(out)], cwd=repo, text=True, capture_output=True)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError(f'Benchmark exited with code {run.returncode}')
manifest = json.loads((out / 'benchmark_manifest.json').read_text(encoding='utf-8'))
c1 = pd.read_csv(out / 'c1_learner_scenarios.csv')
c2 = pd.read_csv(out / 'c2_skill6_text_alignment.csv')
c3 = pd.read_csv(out / 'c3_paired_multimodal_scenarios.csv')
c1_metrics = pd.read_csv(out / 'c1_model_comparison.csv')
c3_metrics = pd.read_csv(out / 'c3_model_comparison.csv')
print('Generated outputs:', out)

## Dataset and provenance inspection

Each generated row carries a synthetic subject ID, dataset ID, split, label, source, and validation status. The nominal train/test split is subject-disjoint; the challenge scenario uses a separately generated cohort.

In [ ]:
display(pd.DataFrame([
    {'component': 'C1', 'rows': len(c1), 'labels': c1.label.nunique(), 'splits': ', '.join(sorted(c1.split.unique())), 'origin': ', '.join(c1.data_origin.unique())},
    {'component': 'C2', 'rows': len(c2), 'labels': c2.perturbation_family.nunique(), 'splits': 'controlled perturbation families', 'origin': ', '.join(c2.data_origin.unique())},
    {'component': 'C3', 'rows': len(c3), 'labels': c3.label.nunique(), 'splits': ', '.join(sorted(c3.split.unique())), 'origin': ', '.join(c3.data_origin.unique())},
]))
display(c1.groupby(['split', 'label']).size().rename('rows').to_frame().head(20))
display(c2[['skill_id', 'activity_id', 'item_id', 'expected_text', 'recognized_text', 'perturbation_family', 'grapheme_error_rate']].head(12))
display(c3[['synthetic_subject_id', 'split', 'label', 'acoustic_latency_ms', 'peak_count_delta', 'time_to_first_touch_ms', 'orthographic_confusion_index']].head())

## C1 model comparison

The unchanged current C1 runtime artifact is scored alongside a majority baseline and retrained comparison candidates. Models are evaluated on held-out learner IDs and a separately generated lower-signal/noisier challenge cohort. Report macro-F1 and balanced accuracy alongside accuracy because class-wise performance matters. These scores are simulator-label recovery, not educational or clinical accuracy.

In [ ]:
display(c1_metrics.style.format({'accuracy': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}))
c1_shift = pd.DataFrame(manifest['c1']['scenario_shift_metrics'])
display(c1_shift.style.format({'accuracy': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}))
best_c1 = c1_metrics.sort_values('macro_f1', ascending=False).iloc[0]['model']
cm = manifest['c1']['nominal'][best_c1]['confusion_matrix']
labels = manifest['c1']['metadata']['label_order']
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set(title=f'C1 held-out confusion matrix — {best_c1}', xlabel='Predicted scenario', ylabel='Generated scenario label', xticks=range(len(labels)), yticks=range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i][j], ha='center', va='center')
fig.colorbar(im, ax=ax); fig.tight_layout()

## C2 Sinhala alignment behavior (Skill 6)

The live alignment service is tested against controlled substitutions, deletions, insertions, repetitions, and mixed edits of textbook prompts. The displayed grapheme accuracy is derived from the service's grapheme error rate. It is **not** an accuracy score for Whisper, STT, speech scoring, or children. Those require consented and independently annotated audio.

In [ ]:
c2_summary = c2.groupby('perturbation_family', as_index=False).agg(
    examples=('item_id', 'count'), mean_word_error_rate=('word_error_rate', 'mean'),
    mean_grapheme_error_rate=('grapheme_error_rate', 'mean'),
    mean_grapheme_accuracy=('grapheme_accuracy', 'mean'), exact_match_rate=('exact_match', 'mean'))
display(c2_summary.style.format({c: '{:.3f}' for c in ['mean_word_error_rate', 'mean_grapheme_error_rate', 'mean_grapheme_accuracy', 'exact_match_rate']}))
ax = c2_summary.set_index('perturbation_family')['mean_grapheme_error_rate'].plot(kind='bar', color='#4a90d9', figsize=(8, 4), title='C2 alignment grapheme error under controlled text edits')
ax.set_ylabel('Mean grapheme error rate'); ax.set_xlabel('Synthetic perturbation family'); plt.xticks(rotation=30, ha='right'); plt.tight_layout()

## C3 candidate model comparison

C3 scores the unchanged current runtime XGBoost artifact alongside a majority baseline and retrained comparison candidates. The joint synthetic modality rows follow current deployed feature names and class order but are not assembled from the C1/C2 datasets, actual recordings, Whisper measurements, or app observations. A separate subject-disjoint test and an independently generated lower-signal/noisier challenge set are reported. This demonstrates schema/model comparison against a shared synthetic scenario label, not production fusion performance.

In [ ]:
display(c3_metrics.style.format({'accuracy': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}))
c3_shift = pd.DataFrame(manifest['c3']['scenario_shift_metrics'])
display(c3_shift.style.format({'accuracy': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}))
best_c3 = c3_metrics.sort_values('macro_f1', ascending=False).iloc[0]['model']
cm3 = manifest['c3']['nominal'][best_c3]['confusion_matrix']
labels3 = manifest['c3']['label_order']
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(cm3, cmap='Purples')
ax.set(title=f'C3 held-out confusion matrix — {best_c3}', xlabel='Predicted scenario', ylabel='Generated scenario label', xticks=range(len(labels3)), yticks=range(len(labels3)))
ax.set_xticklabels(labels3, rotation=45, ha='right'); ax.set_yticklabels(labels3)
for i in range(len(labels3)):
    for j in range(len(labels3)):
        ax.text(j, i, cm3[i][j], ha='center', va='center')
fig.colorbar(im, ax=ax); fig.tight_layout()

## Standards-informed checklist and viva defense

- **Provenance and scope:** manifest states fully synthetic origin, generator/version, fixed seed, feature schema, and class-generation rules.
- **Data quality:** inspect missingness, ranges, class counts, activity/skill coverage, and whether the synthetic features match the live extractor contract. Current C1 calls the same feature builder as runtime; C3 checks the current feature-name contract.
- **Evaluation integrity:** hold out subjects, publish class-wise metrics/confusion matrices, compare against simple baselines, and separately report a shifted scenario.
- **Privacy and validity:** no real child records are used. Synthetic generation does not establish privacy guarantees, representativeness, or clinical validity; the generated labels encode assumptions.
- **Reproducibility:** record seeds and versions; keep research CSVs and comparisons outside deployed model directories.
- **C2-specific:** controlled Sinhala text alignment checks the algorithm and orthographic-aware scoring only. A real STT evaluation needs representative consented recordings and independent transcript annotation.

Suggested viva wording: ‘These are synthetic scenario-recovery experiments. The split prevents the same simulated learner appearing in train and test, and the challenge set probes sensitivity to changed simulation conditions. This validates implementation behavior against explicit assumptions; it does not claim performance on Sri Lankan children or diagnose dyslexia. We will calibrate and externally evaluate only with appropriate ethics approval, consent, and expert-reviewed data.’

The reporting choices are informed by [ISO/IEC 5259](https://www.iso.org/standard/81088.html), [NIST SDNist](https://www.nist.gov/services-resources/software/sdnist-synthetic-data-report-tool), and [TRIPOD+AI](https://www.tripod-statement.org/scope/). The project is not claiming conformance or certification.

In [ ]:
print('Manifest summary')
print(json.dumps({k: manifest[k] for k in ['dataset_id', 'created_at_utc', 'seed', 'data_origin', 'interpretation_limits', 'standards_guidance']}, ensure_ascii=False, indent=2))
print('C1 split metadata:', manifest['c1']['metadata'])
print('C3 split metadata:', {k: manifest['c3'][k] for k in ['subject_overlap_train_test', 'train_rows', 'test_rows', 'challenge_rows']})